# Phase 8 — Feature Engineering

## Task 8.1 — Feature Ideas

The purpose of feature engineering is to transform the analytical dataset into a feature set that represents content performance, search exposure, and content-level characteristics in a form suitable for the opportunity-scoring problem.

Feature engineering will focus on creating meaningful and interpretable variables rather than maximizing the number of features.

### 1. Search Performance Features

#### `total_gsc_impressions`

**Definition:** Total Google Search impressions observed during the reporting period.

**Reason:** Represents the observed search visibility of the content item.

**Expected treatment:** Because impressions are highly right-skewed, a transformed version may be considered.

---

#### `total_gsc_clicks`

**Definition:** Total Google Search clicks observed during the reporting period.

**Reason:** Represents observed search traffic generated by the content item.

**Expected treatment:** The strong concentration of zero values and extreme observations should be considered during transformation.

---

#### `mean_gsc_avg_position`

**Definition:** Mean observed Google Search average position.

**Reason:** Provides information about the content item's observed search ranking position.

**Expected treatment:** Missing values require intentional handling because missing position is strongly associated with zero impressions.

---

### 2. Click-Through Features

#### `ctr`

**Definition:**

CTR is calculated as:

`total_gsc_clicks / total_gsc_impressions`

for content items with impressions greater than zero.

**Reason:** Represents the proportion of impressions that resulted in clicks.

**Expected treatment:** Low-exposure observations should be interpreted carefully because CTR can be unstable when impression counts are small.

---

### 3. Exposure Features

#### `reporting_days`

**Definition:** Number of distinct reporting days available for the content item.

**Reason:** Represents the amount of observed exposure available for evaluating the item's historical performance.

---

#### `impressions_per_reporting_day`

**Definition:**

`total_gsc_impressions / reporting_days`

**Reason:** Normalizes total impressions by the number of reporting days and makes content items with different reporting coverage more comparable.

---

#### `clicks_per_reporting_day`

**Definition:**

`total_gsc_clicks / reporting_days`

**Reason:** Normalizes observed clicks according to the available reporting period.

---

### 4. Content Size Features

#### `word_count`

**Definition:** Number of words in the content item.

**Reason:** Represents content size.

**Expected treatment:** The feature contains missing values and a wide distribution, so transformation or missing-value handling may be required.

---

#### `char_count`

**Definition:** Number of characters in the content item.

**Reason:** Provides an additional measure of content size.

**Expected treatment:** Because `char_count` and `word_count` measure related properties, their redundancy should be evaluated.

---

### 5. Search Demand and Competition Features

#### `search_volume`

**Definition:** Associated search volume for the content item.

**Reason:** Represents the available search demand associated with the content.

**Expected treatment:** Missing values and skewness should be investigated.

---

#### `competition`

**Definition:** Numerical competition measure associated with the content.

**Reason:** Represents the competitive search environment.

**Expected treatment:** Distribution and relationship with `competition_level` should be investigated.

---

#### `competition_level`

**Definition:** Categorical representation of search competition.

**Reason:** Provides a categorical representation of the competitive environment.

**Expected treatment:** Categorical encoding will be considered during implementation.

---

### 6. Content Characteristics

#### `content_type`

**Definition:** Type of content, such as keyword article, Feedly article, or comparison article.

**Reason:** Different content types show different historical performance distributions.

**Expected treatment:** Categorical encoding.

---

#### `main_intent`

**Definition:** Main search intent associated with the content.

**Reason:** Search intent may help represent the relationship between content purpose and observed search performance.

**Expected treatment:** Missing values will be handled intentionally and encoded as a meaningful category where appropriate.

---

#### `category_count`

**Definition:** Number of categories associated with the content item.

**Reason:** Represents the breadth of content categorization.

**Expected treatment:** Distribution will be checked before deciding whether transformation is necessary.

---

### 7. Backlink Features

#### `backlinks`

**Definition:** Number of backlinks associated with the content item.

**Reason:** Represents an external authority-related characteristic that may be associated with search performance.

**Expected treatment:** The feature is highly right-skewed and contains both zero and missing values. Transformation and missing-value handling will therefore be considered.

---

### Feature Engineering Principles

The following principles will guide feature construction:

1. Features must have a clear interpretation.
2. Features must be reproducible from the analytical dataset.
3. Features must use only information legitimately available at prediction time.
4. Highly skewed numerical variables may require transformation.
5. Missing values will be handled according to their meaning rather than removed automatically.
6. Redundant features will be reviewed before final model training.
7. Feature engineering will remain intentionally simple and interpretable.
8. Features will not be created solely to improve model performance without a clear analytical justification.

### Self-check

Before you submit, confirm each line honestly:

- [x] I listed candidate features before implementing them
- [x] Each feature has a clear meaning
- [x] Each feature has a reason for being useful
- [x] Features are derived from information legitimately available at prediction time
- [x] I avoided creating features just to increase model performance

In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute("PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp/duckdb_tmp';")
con.execute("PRAGMA memory_limit='1GB';")

In [1]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

In [4]:
con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")

In [5]:
con.execute("""
    CREATE OR REPLACE TABLE performance_aggregated AS
    SELECT
        client_hash_id,
        content_hash_id,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS reporting_days,
        SUM(gsc_impressions) AS total_gsc_impressions,
        SUM(gsc_clicks) AS total_gsc_clicks,
        SUM(gsc_sum_position) AS total_gsc_sum_position,
        AVG(gsc_avg_position) AS mean_gsc_avg_position
    FROM performance_deduplicated
    GROUP BY client_hash_id, content_hash_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
con.execute(f"""
    CREATE OR REPLACE TABLE analytical_dataset AS
    SELECT
        p.client_hash_id,
        p.content_hash_id,

        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted,

        p.first_report_date,
        p.last_report_date,
        p.reporting_days,
        p.total_gsc_impressions,
        p.total_gsc_clicks,
        p.total_gsc_sum_position,
        p.mean_gsc_avg_position

    FROM performance_aggregated p
    INNER JOIN read_parquet('{content_file}') c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""")

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE feature_dataset AS

    SELECT
        client_hash_id,
        content_hash_id,

        -- Performance features
        total_gsc_impressions,
        total_gsc_clicks,
        mean_gsc_avg_position,
        reporting_days,

        -- CTR
        CASE
            WHEN total_gsc_impressions > 0
            THEN total_gsc_clicks * 1.0 / total_gsc_impressions
            ELSE NULL
        END AS ctr,

        -- Exposure-normalized features
        CASE
            WHEN reporting_days > 0
            THEN total_gsc_impressions * 1.0 / reporting_days
            ELSE NULL
        END AS impressions_per_reporting_day,

        CASE
            WHEN reporting_days > 0
            THEN total_gsc_clicks * 1.0 / reporting_days
            ELSE NULL
        END AS clicks_per_reporting_day,

        -- Search demand / competition
        search_volume,
        competition,
        competition_level,
        cpc,

        -- Content characteristics
        content_type,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count,

        -- Status / timing
        last_optimized_date,
        optimization_eligible_date,
        is_published,
        is_deleted

    FROM analytical_dataset
""")

In [9]:
con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN log_impressions DOUBLE
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN log_clicks DOUBLE
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN log_backlinks DOUBLE
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN log_search_volume DOUBLE
""")

con.execute("""
    UPDATE feature_dataset
    SET
        log_impressions = LN(1 + total_gsc_impressions),
        log_clicks = LN(1 + total_gsc_clicks),
        log_backlinks = CASE
            WHEN backlinks IS NOT NULL
            THEN LN(1 + backlinks)
            ELSE NULL
        END,
        log_search_volume = CASE
            WHEN search_volume IS NOT NULL
            THEN LN(1 + search_volume)
            ELSE NULL
        END
""")

In [ ]:
# -------------------------
# ----Missing Indicators---
# -------------------------

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_search_volume INTEGER
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_main_intent INTEGER
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_competition_level INTEGER
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_backlinks INTEGER
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_word_count INTEGER
""")

con.execute("""
    ALTER TABLE feature_dataset
    ADD COLUMN missing_char_count INTEGER
""")

In [11]:
con.execute("""
    UPDATE feature_dataset
    SET
        missing_search_volume =
            CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END,

        missing_main_intent =
            CASE WHEN main_intent IS NULL THEN 1 ELSE 0 END,

        missing_competition_level =
            CASE WHEN competition_level IS NULL THEN 1 ELSE 0 END,

        missing_backlinks =
            CASE WHEN backlinks IS NULL THEN 1 ELSE 0 END,

        missing_word_count =
            CASE WHEN word_count IS NULL THEN 1 ELSE 0 END,

        missing_char_count =
            CASE WHEN char_count IS NULL THEN 1 ELSE 0 END
""")

In [12]:
# ------------------------------------
# Check the Resulting Feature Dataset
# ------------------------------------

feature_shape = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_content_client_pairs
    FROM feature_dataset
""").df()

feature_shape

,total_rows,unique_content_client_pairs
0,409205,409205


In [15]:
feature_dataset = con.execute("""
    SELECT *
    FROM feature_dataset
    LIMIT 5
""").df()

feature_dataset.head()

,client_hash_id,content_hash_id,total_gsc_impressions,total_gsc_clicks,mean_gsc_avg_position,reporting_days,ctr,impressions_per_reporting_day,clicks_per_reporting_day,search_volume,...,log_impressions,log_clicks,log_backlinks,log_search_volume,missing_search_volume,missing_main_intent,missing_competition_level,missing_backlinks,missing_word_count,missing_char_count
0,client_1a8bf67cad4ee525,content_66815be25c2aabad,961.0,8.0,5.371188,30,0.008325,32.033333,0.266667,0,...,6.869014,2.197225,0.0,0.000000,0,0,0,0,0,0
1,client_1a8bf67cad4ee525,content_66a558a4c3e9df46,400.0,2.0,2.233805,30,0.005000,13.333333,0.066667,10,...,5.993961,1.098612,0.0,2.397895,0,0,0,0,0,0
2,client_1a8bf67cad4ee525,content_66c3f7503109aebd,165.0,0.0,3.417626,30,0.000000,5.500000,0.000000,10,...,5.111988,0.000000,0.0,2.397895,0,0,0,0,0,0
3,client_1a8bf67cad4ee525,content_66dadf5510782af7,321.0,0.0,2.019991,30,0.000000,10.700000,0.000000,0,...,5.774552,0.000000,0.0,0.000000,0,0,0,0,0,0
4,client_1a8bf67cad4ee525,content_67094ffd228da906,237.0,0.0,54.990292,20,0.000000,11.850000,0.000000,0,...,5.472271,0.000000,0.0,0.000000,0,0,0,0,0,0


In [16]:
con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM feature_dataset
""").df()

,total_rows
0,409205


### Implemented Features

The feature dataset was created from the analytical dataset while preserving the original analytical grain of:

`client_hash_id + content_hash_id`

The implemented features include:

#### Performance Features

- `total_gsc_impressions`
- `total_gsc_clicks`
- `mean_gsc_avg_position`
- `reporting_days`

#### Derived Performance Features

- `ctr`
- `impressions_per_reporting_day`
- `clicks_per_reporting_day`

#### Transformed Numerical Features

- `log_impressions`
- `log_clicks`
- `log_backlinks`
- `log_search_volume`

The logarithmic transformation uses:

`LN(1 + x)`

to handle zero-valued observations safely.

#### Missingness Indicators

- `missing_search_volume`
- `missing_main_intent`
- `missing_competition_level`
- `missing_backlinks`
- `missing_word_count`
- `missing_char_count`

These indicators preserve information about whether important variables were unavailable.

#### Content and Search Features

- `search_volume`
- `competition`
- `competition_level`
- `cpc`
- `content_type`
- `main_intent`
- `backlinks`
- `category_count`
- `char_count`
- `word_count`

The original analytical dataset was not modified. A separate `feature_dataset` was created to preserve reproducibility and allow comparison between raw and engineered variables.

In [ ]:
# -----------------------------
# ---Validate Feature Ranges---
# -----------------------------
numeric_validation = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        MIN(total_gsc_impressions) AS min_impressions,
        MAX(total_gsc_impressions) AS max_impressions,

        MIN(total_gsc_clicks) AS min_clicks,
        MAX(total_gsc_clicks) AS max_clicks,

        MIN(ctr) AS min_ctr,
        MAX(ctr) AS max_ctr,

        MIN(reporting_days) AS min_reporting_days,
        MAX(reporting_days) AS max_reporting_days,

        MIN(impressions_per_reporting_day) AS min_impressions_per_day,
        MAX(impressions_per_reporting_day) AS max_impressions_per_day,

        MIN(clicks_per_reporting_day) AS min_clicks_per_day,
        MAX(clicks_per_reporting_day) AS max_clicks_per_day,

        MIN(word_count) AS min_word_count,
        MAX(word_count) AS max_word_count,

        MIN(char_count) AS min_char_count,
        MAX(char_count) AS max_char_count,

        MIN(backlinks) AS min_backlinks,
        MAX(backlinks) AS max_backlinks,

        MIN(category_count) AS min_category_count,
        MAX(category_count) AS max_category_count
    FROM feature_dataset
""").df()

numeric_validation

,total_rows,min_impressions,max_impressions,min_clicks,max_clicks,min_ctr,max_ctr,min_reporting_days,max_reporting_days,min_impressions_per_day,...,min_clicks_per_day,max_clicks_per_day,min_word_count,max_word_count,min_char_count,max_char_count,min_backlinks,max_backlinks,min_category_count,max_category_count
0,409205,0.0,615012.0,0.0,152170.0,0.0,1.0,3,30,0.0,...,0.0,5072.333333,0,29341,0,357575,0,4269360,0,18


In [18]:
# -----------------------------
# ---Check Unexpected Values---
# -----------------------------
validation_checks = con.execute("""
    SELECT
        SUM(CASE WHEN total_gsc_impressions < 0 THEN 1 ELSE 0 END)
            AS negative_impressions,

        SUM(CASE WHEN total_gsc_clicks < 0 THEN 1 ELSE 0 END)
            AS negative_clicks,

        SUM(CASE WHEN ctr < 0 OR ctr > 1 THEN 1 ELSE 0 END)
            AS invalid_ctr,

        SUM(CASE WHEN reporting_days <= 0 THEN 1 ELSE 0 END)
            AS invalid_reporting_days,

        SUM(CASE WHEN impressions_per_reporting_day < 0 THEN 1 ELSE 0 END)
            AS negative_impressions_per_day,

        SUM(CASE WHEN clicks_per_reporting_day < 0 THEN 1 ELSE 0 END)
            AS negative_clicks_per_day,

        SUM(CASE WHEN word_count < 0 THEN 1 ELSE 0 END)
            AS negative_word_count,

        SUM(CASE WHEN char_count < 0 THEN 1 ELSE 0 END)
            AS negative_char_count,

        SUM(CASE WHEN backlinks < 0 THEN 1 ELSE 0 END)
            AS negative_backlinks,

        SUM(CASE WHEN category_count < 0 THEN 1 ELSE 0 END)
            AS negative_category_count

    FROM feature_dataset
""").df()

validation_checks

,negative_impressions,negative_clicks,invalid_ctr,invalid_reporting_days,negative_impressions_per_day,negative_clicks_per_day,negative_word_count,negative_char_count,negative_backlinks,negative_category_count
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# -----------------------------
# ---Validate Missing Values---
# -----------------------------

missing_validation = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END)
            AS missing_search_volume,

        SUM(CASE WHEN main_intent IS NULL THEN 1 ELSE 0 END)
            AS missing_main_intent,

        SUM(CASE WHEN competition_level IS NULL THEN 1 ELSE 0 END)
            AS missing_competition_level,

        SUM(CASE WHEN backlinks IS NULL THEN 1 ELSE 0 END)
            AS missing_backlinks,

        SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END)
            AS missing_word_count,

        SUM(CASE WHEN char_count IS NULL THEN 1 ELSE 0 END)
            AS missing_char_count,

        SUM(CASE WHEN mean_gsc_avg_position IS NULL THEN 1 ELSE 0 END)
            AS missing_avg_position,

        SUM(CASE WHEN ctr IS NULL THEN 1 ELSE 0 END)
            AS missing_ctr

    FROM feature_dataset
""").df()

missing_validation

,total_rows,missing_search_volume,missing_main_intent,missing_competition_level,missing_backlinks,missing_word_count,missing_char_count,missing_avg_position,missing_ctr
0,409205,66001.0,71511.0,67831.0,175929.0,107758.0,107758.0,200569.0,200569.0


In [20]:
# -------------------------------
# Validate the Missing Indicators
# -------------------------------

indicator_validation = con.execute("""
    SELECT
        SUM(
            CASE
                WHEN missing_search_volume =
                     CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END
                THEN 0 ELSE 1
            END
        ) AS search_volume_indicator_errors,

        SUM(
            CASE
                WHEN missing_main_intent =
                     CASE WHEN main_intent IS NULL THEN 1 ELSE 0 END
                THEN 0 ELSE 1
            END
        ) AS main_intent_indicator_errors,

        SUM(
            CASE
                WHEN missing_backlinks =
                     CASE WHEN backlinks IS NULL THEN 1 ELSE 0 END
                THEN 0 ELSE 1
            END
        ) AS backlinks_indicator_errors,

        SUM(
            CASE
                WHEN missing_word_count =
                     CASE WHEN word_count IS NULL THEN 1 ELSE 0 END
                THEN 0 ELSE 1
            END
        ) AS word_count_indicator_errors,

        SUM(
            CASE
                WHEN missing_char_count =
                     CASE WHEN char_count IS NULL THEN 1 ELSE 0 END
                THEN 0 ELSE 1
            END
        ) AS char_count_indicator_errors

    FROM feature_dataset
""").df()

indicator_validation

,search_volume_indicator_errors,main_intent_indicator_errors,backlinks_indicator_errors,word_count_indicator_errors,char_count_indicator_errors
0,0.0,0.0,0.0,0.0,0.0


In [21]:
# ------------------------------
# Check Feature Transformations
# ------------------------------

transformation_check = con.execute("""
    SELECT
        MIN(log_impressions) AS min_log_impressions,
        MAX(log_impressions) AS max_log_impressions,

        MIN(log_clicks) AS min_log_clicks,
        MAX(log_clicks) AS max_log_clicks,

        MIN(log_backlinks) AS min_log_backlinks,
        MAX(log_backlinks) AS max_log_backlinks,

        MIN(log_search_volume) AS min_log_search_volume,
        MAX(log_search_volume) AS max_log_search_volume
    FROM feature_dataset
""").df()

transformation_check

,min_log_impressions,max_log_impressions,min_log_clicks,max_log_clicks,min_log_backlinks,max_log_backlinks,min_log_search_volume,max_log_search_volume
0,0.0,13.329399,0.0,11.93276,0.0,15.266975,0.0,12.815841


In [22]:
con.execute("""
    SELECT
        total_gsc_impressions,
        log_impressions,
        total_gsc_clicks,
        log_clicks,
        backlinks,
        log_backlinks
    FROM feature_dataset
    WHERE total_gsc_impressions IN (0, 1)
       OR total_gsc_clicks IN (0, 1)
       OR backlinks IN (0, 1)
    LIMIT 10
""").df()

,total_gsc_impressions,log_impressions,total_gsc_clicks,log_clicks,backlinks,log_backlinks
0,961.0,6.869014,8.0,2.197225,0,0.000000
1,400.0,5.993961,2.0,1.098612,0,0.000000
2,165.0,5.111988,0.0,0.000000,0,0.000000
3,321.0,5.774552,0.0,0.000000,0,0.000000
4,237.0,5.472271,0.0,0.000000,0,0.000000
5,83.0,4.430817,0.0,0.000000,0,0.000000
6,53.0,3.988984,0.0,0.000000,0,0.000000
7,134.0,4.905275,1.0,0.693147,658,6.490724
8,17.0,2.890372,0.0,0.000000,0,0.000000
9,0.0,0.000000,0.0,0.000000,0,0.000000


### Redundancy Review

Several engineered features represent related information and may therefore contain redundant signals.

Examples include:

- `word_count` and `char_count`
- `competition` and `competition_level`
- `total_gsc_impressions` and `impressions_per_reporting_day`
- `total_gsc_clicks` and `clicks_per_reporting_day`

These features are retained during the validation stage because they have different interpretations and may be useful under different modeling approaches.

Their redundancy and contribution will be reviewed before final model training.

No feature is removed solely because it is correlated with another feature.

### Leakage Check

The engineered features were reviewed for potential data leakage.

The feature dataset does not use:

- Future performance after the evaluation period
- Post-optimization outcomes
- A future success label
- `content_hash_id` or `client_hash_id` as predictive numerical features

The performance features represent the historical observation period used by the project.

Any future modeling split must preserve the temporal structure of the data when required.

The final target/reference-score construction will also be reviewed separately to avoid using target-derived information incorrectly as an independent model feature.

In [23]:
feature_summary = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items
    FROM feature_dataset
""").df()

feature_summary

,total_rows,unique_clients,unique_content_items
0,409205,65,409205


In [24]:
feature_columns = con.execute("""
    DESCRIBE feature_dataset
""").df()

feature_columns[['column_name', 'column_type']]

,column_name,column_type
0,client_hash_id,VARCHAR
1,content_hash_id,VARCHAR
2,total_gsc_impressions,HUGEINT
3,total_gsc_clicks,HUGEINT
4,mean_gsc_avg_position,DOUBLE
5,reporting_days,BIGINT
6,ctr,DOUBLE
7,impressions_per_reporting_day,DOUBLE
8,clicks_per_reporting_day,DOUBLE
9,search_volume,BIGINT


## Task 8.3 — Validate Features

### Validation Results

The engineered feature dataset was successfully validated.

The validation confirmed that:

- The feature dataset contains 409,205 rows.
- The intended `client_hash_id + content_hash_id` grain was preserved.
- No duplicate content-client pairs were introduced.
- Numerical features were checked for invalid negative values.
- CTR values were within the expected 0–1 range.
- Reporting days were within the observed 3–30 day range.
- Exposure-normalized features contained no negative values.
- Word count, character count, backlinks, and category count contained no negative values.
- Missing values were retained intentionally where appropriate.
- Missingness indicators were verified and contained no calculation errors.
- Log transformations were checked for zero and positive values and were calculated correctly using `LN(1 + x)`.
- Potentially redundant feature groups were identified for later review.
- Identifier columns were retained for tracking and ranking but are not intended to be used as predictive features.
- No future or post-outcome information was intentionally introduced during feature construction.

### Feature Dataset Structure

The resulting feature dataset contains the original analytical variables together with engineered performance, exposure, transformation, and missingness features.

The dataset currently contains 33 columns, including:

- Identifier columns
- Historical performance features
- Exposure features
- Content characteristics
- Search and competition features
- Log-transformed numerical features
- Missingness indicators

The final predictive feature subset will be selected before model training after reviewing feature redundancy and the requirements of the selected modeling approach.

### Conclusion

The feature engineering process is reproducible, the resulting dataset preserves the expected analytical grain, and the engineered features have passed the initial validation checks.

The feature dataset is ready for the modeling stage.

### Self-check

Before you submit, confirm each line honestly:

- [x] Feature ranges were checked
- [x] Unexpected values were investigated
- [x] Leakage was checked again
- [x] Highly redundant features were considered
- [x] Feature calculations were manually verified for selected rows
- [x] The final feature set is documented